**When this Date Skew Case will Occur?**



*   When the (post shuffle result) shuffle ends with unevenly distributed data (a "Skewed" result), AQE steps in like a load balancer.
*   It detects the imbalance before the next stage starts and dynamically fixes it


Before understanding Adaptive execution, you must know how Spark usually works.


1.   Logical Plan: You write code; Spark makes a "to-do list."
2.   Physical Plan: Spark decides how to do it (e.g., using a SortMergeJoin).

**The Problem**: Once the job starts, the plan is locked. If the data is much bigger or smaller than expected, Spark doesn't change its mind, leading to slow performance or "Out of Memory" errors.

**What Happens when AQE set as True?**

*  When set to true, Spark re-optimizes the query plan between stages based on runtime statistics.

Syntax: # How to enable it in your Spark Session

spark.conf.set("spark.sql.adaptive.enabled", "true")


# Simple Demo on Data Skew

In [ ]:
import pyspark

In [ ]:
from pyspark.sql import SparkSession

# Stop existing session if one exists
if 'spark' in locals():
    spark.stop()

# Start new session with higher memory (e.g., 8GB)
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.ui.port", "4050") \
    .getOrCreate()
print(spark.version)
# spark.stop()

4.0.2


In [ ]:

aqeVal_before=spark.conf.get("spark.sql.adaptive.enabled")
print("Before Config: AQE - ",aqeVal_before)
spark.conf.set("spark.sql.adaptive.enabled", "false")
aqeVal_after=spark.conf.get("spark.sql.adaptive.enabled")
print("After Config: AQE - ",aqeVal_after)

Before Config: AQE -  true
After Config: AQE -  false


In [ ]:
#BroadcastHashJoin
broadCastVal=spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
print("Before Config: BroadCast Threshold - ",broadCastVal)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
broadCastVal=spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
print("After Config: BroadCast Threshold - ",broadCastVal)

Before Config: BroadCast Threshold -  10485760b
After Config: BroadCast Threshold -  -1


In [ ]:
from pyspark.sql.functions import col, explode, lit, sequence
from pyspark.sql import functions as F

# 1. Create a skewed dataset (Key '100' appears 1 million times!)
#    Other keys (1-99) appear only once.
skewed_df = spark.range(0, 1).select(
    explode(
        sequence(lit(1), lit(1000000))
    ).alias("id")
).withColumn("key", lit(100)) \
.union(
    spark.range(1, 100).toDF("temp_id")
        .withColumn("key", F.col("temp_id"))
        .withColumnRenamed("temp_id", "id")
)

# 2. Create a small lookup table (Product Names)
small_df = spark.range(1, 105).toDF("key").withColumn("name", lit("Product_Name"))

In [ ]:
skewed_df.orderBy(col('id').desc()).show(5)
small_df.show(5)

+-------+---+
|     id|key|
+-------+---+
|1000000|100|
| 999999|100|
| 999998|100|
| 999997|100|
| 999996|100|
+-------+---+
only showing top 5 rows
+---+------------+
|key|        name|
+---+------------+
|  1|Product_Name|
|  2|Product_Name|
|  3|Product_Name|
|  4|Product_Name|
|  5|Product_Name|
+---+------------+
only showing top 5 rows


As you can see in the Physical Plan output, Spark is now performing a SortMergeJoin.

This means that both skewed_df and small_df are being shuffled based on their key column (Exchange hashpartitioning). Since the key = 100 in skewed_df has a million rows, this specific partition will be significantly larger than others after the shuffle, leading to severe data skew. The SortMergeJoin then attempts to sort and merge these partitions, and the skewed partition will become a bottleneck, making the join very inefficient.

In [ ]:
joined_df = skewed_df.join(small_df, "key")

# Inspect the Plan
joined_df.explain()

== Physical Plan ==
*(6) Project [key#9L, id#8L, Product_Name AS name#12]
+- *(6) SortMergeJoin [key#9L], [key#11L], Inner
   :- *(3) Sort [key#9L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(key#9L, 200), ENSURE_REQUIREMENTS, [plan_id=99]
   :     +- Union
   :        :- *(1) Project [cast(id#2 as bigint) AS id#8L, 100 AS key#9L]
   :        :  +- *(1) Generate explode(org.apache.spark.sql.catalyst.expressions.UnsafeArrayData@d8890950), false, [id#2]
   :        :     +- *(1) Project
   :        :        +- *(1) Range (0, 1, step=1, splits=2)
   :        +- *(2) Project [id#4L, id#4L AS key#6L]
   :           +- *(2) Range (1, 100, step=1, splits=2)
   +- *(5) Sort [key#11L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(key#11L, 200), ENSURE_REQUIREMENTS, [plan_id=105]
         +- *(4) Project [id#10L AS key#11L]
            +- *(4) Range (1, 105, step=1, splits=2)




In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

# FORCE AQE to trigger by invoking an action (count)
# AQE only kicks in *during* execution, not during definition.
joined_df.count()

# Inspect the Plan (Adaptive Plan)
joined_df.explain()

== Physical Plan ==
*(6) Project [key#9L, id#8L, Product_Name AS name#12]
+- *(6) SortMergeJoin [key#9L], [key#11L], Inner
   :- *(3) Sort [key#9L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(key#9L, 200), ENSURE_REQUIREMENTS, [plan_id=99]
   :     +- Union
   :        :- *(1) Project [cast(id#2 as bigint) AS id#8L, 100 AS key#9L]
   :        :  +- *(1) Generate explode(org.apache.spark.sql.catalyst.expressions.UnsafeArrayData@d8890950), false, [id#2]
   :        :     +- *(1) Project
   :        :        +- *(1) Range (0, 1, step=1, splits=2)
   :        +- *(2) Project [id#4L, id#4L AS key#6L]
   :           +- *(2) Range (1, 100, step=1, splits=2)
   +- *(5) Sort [key#11L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(key#11L, 200), ENSURE_REQUIREMENTS, [plan_id=105]
         +- *(4) Project [id#10L AS key#11L]
            +- *(4) Range (1, 105, step=1, splits=2)




### Inspecting Runtime Plan with `toDebugString()`

To see what Spark is actually doing under the hood after AQE has had a chance to optimize, we can trigger an action and then inspect the RDD's debug string. Note that this is a lower-level view than `explain()`.

In [ ]:
# Trigger an action to force AQE optimizations
# This will run the query and allow AQE to make its decisions
print("Triggering action with count()...")
joined_df.count()

# Now, inspect the RDD lineage of the joined_df
# This gives a detailed, low-level view of the physical operations
print("\nRDD Debug String:")
print(joined_df.rdd.toDebugString().decode('utf-8'))

Triggering action with count()...

RDD Debug String:
(200) MapPartitionsRDD[77] at javaToPython at NativeMethodAccessorImpl.java:0 []
  |   MapPartitionsRDD[76] at javaToPython at NativeMethodAccessorImpl.java:0 []
  |   SQLExecutionRDD[75] at javaToPython at NativeMethodAccessorImpl.java:0 []
  |   MapPartitionsRDD[74] at javaToPython at NativeMethodAccessorImpl.java:0 []
  |   ZippedPartitionsRDD2[73] at javaToPython at NativeMethodAccessorImpl.java:0 []
  |   MapPartitionsRDD[66] at javaToPython at NativeMethodAccessorImpl.java:0 []
  |   ShuffledRowRDD[65] at javaToPython at NativeMethodAccessorImpl.java:0 []
  +-(4) MapPartitionsRDD[64] at javaToPython at NativeMethodAccessorImpl.java:0 []
     |  UnionRDD[63] at javaToPython at NativeMethodAccessorImpl.java:0 []
     |  MapPartitionsRDD[59] at javaToPython at NativeMethodAccessorImpl.java:0 []
     |  MapPartitionsRDD[58] at javaToPython at NativeMethodAccessorImpl.java:0 []
     |  ParallelCollectionRDD[57] at javaToPython at Na

When `spark.sql.adaptive.enabled` is `true`, the `joined_df.explain()` output will often start with `AdaptiveSparkPlan isFinalPlan=false`. This indicates that the plan is still undergoing runtime optimization. After an action is triggered and the query completes, if you were to re-run `joined_df.explain()`, it would then show `isFinalPlan=true` and potentially reflect the final, optimized plan, although the detailed skew handling might still be subtle.

The most comprehensive way to observe AQE's specific actions, such as dynamically handling skew, re-optimizing joins, or coalescing partitions, is to monitor the **Spark UI** during the execution of an action. The Event Timeline and the details within the SQL tab for the executed query will provide visual cues and metrics on how AQE adjusted the plan.

In Spark, it's often used as the spark.sql.autoBroadcastJoinThreshold configuration. This setting determines the maximum size (in bytes) of a table that Spark will broadcast to all worker nodes when performing a join. If one of the tables in a join is smaller than this threshold, Spark can choose to broadcast it, which can significantly speed up the join by avoiding a shuffle.

The calculation is 10 * 1024 * 1024:

    1024 bytes = 1 kilobyte (KB)
    1024 KB = 1 megabyte (MB)

So, 10 * 1024 * 1024 = 10 * 1MB = 10MB.

In [ ]:
# Perform the Join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760") # Enable broadcast for this example (10MB)
joined_df = skewed_df.join(small_df, "key")

# Inspect the Plan
joined_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [key#9L, id#8L, Product_Name AS name#12]
   +- BroadcastHashJoin [key#9L], [key#11L], Inner, BuildRight, false
      :- Union
      :  :- Project [cast(id#2 as bigint) AS id#8L, 100 AS key#9L]
      :  :  +- Generate explode(org.apache.spark.sql.catalyst.expressions.UnsafeArrayData@d8890950), false, [id#2]
      :  :     +- Project
      :  :        +- Range (0, 1, step=1, splits=2)
      :  +- Project [id#4L, id#4L AS key#6L]
      :     +- Range (1, 100, step=1, splits=2)
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=607]
         +- Project [id#10L AS key#11L]
            +- Range (1, 105, step=1, splits=2)




# Next Example

In [ ]:
# Disable automatic optimizations to see the skew clearly
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

In [ ]:
# Check Spark context and executor info
print("Spark Version:", spark.version)
print("App Name:", spark.sparkContext.appName)

# Check executor memory config
print("\n--- Memory Configurations ---")
print("executor.memory:", spark.conf.get("spark.executor.memory", "not set"))
print("driver.memory:", spark.conf.get("spark.driver.memory", "not set"))
print("sql.shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("default.parallelism:", spark.conf.get("spark.default.parallelism", "not set"))

Spark Version: 4.0.2
App Name: pyspark-shell

--- Memory Configurations ---
executor.memory: not set
driver.memory: 8g
sql.shuffle.partitions: 200
default.parallelism: not set


In [ ]:
# Serverless-safe cluster/config inspection
print("Spark Version:", spark.version)

# Safe config reads — serverless may return "not set" for JVM-level configs, that's expected
configs_to_check = [
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.files.maxPartitionBytes",
    "spark.sql.autoBroadcastJoinThreshold"
]

print("\n--- Spark SQL Configurations ---")
for cfg in configs_to_check:
    try:
        val = spark.conf.get(cfg)
        print(f"{cfg}: {val}")
    except Exception:
        print(f"{cfg}: not set")

Spark Version: 4.0.2

--- Spark SQL Configurations ---
spark.sql.shuffle.partitions: 200
spark.sql.adaptive.enabled: false
spark.sql.adaptive.skewJoin.enabled: true
spark.sql.files.maxPartitionBytes: 134217728b
spark.sql.autoBroadcastJoinThreshold: -1


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import random

# Simulate a sales dataset with skewed customer_id distribution
# customer_id = 1 has 80% of all records (the hot key)

random.seed(42)

def generate_skewed_data(n=2_000_000):
    rows = []
    for i in range(n):
        r = random.random()
        if r < 0.80:
            customer_id = 1          # HOT KEY — 80% of data lands here
        elif r < 0.90:
            customer_id = 2          # 10%
        elif r < 0.95:
            customer_id = 3          # 5%
        else:
            customer_id = random.randint(4, 1000)  # remaining 5% spread across 996 customers

        rows.append((customer_id, random.randint(1, 500), round(random.uniform(10, 5000), 2)))
    return rows

In [ ]:
data = generate_skewed_data(2_000_000)
schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("product_id",  IntegerType(), False),
    StructField("amount",      DoubleType(),  False)
])

df = spark.createDataFrame(data, schema)
df.cache()
df.count()  # trigger cache

print("Total rows:", df.count())
df.show(5)

Total rows: 2000000
+-----------+----------+-------+
|customer_id|product_id| amount|
+-----------+----------+-------+
|          1|        13|3710.34|
|          1|        72|3684.99|
|          1|       457|2731.38|
|          1|        17| 158.69|
|          1|       259|3014.07|
+-----------+----------+-------+
only showing top 5 rows


In [ ]:
# Visualize how data is distributed
skew_distribution = df.groupBy("customer_id") \
    .count() \
    .withColumnRenamed("count", "row_count") \
    .orderBy(F.desc("row_count"))

skew_distribution.show(10)


+-----------+---------+
|customer_id|row_count|
+-----------+---------+
|          1|  1600514|
|          2|   199697|
|          3|    99949|
|        619|      128|
|        959|      128|
|        292|      126|
|        912|      126|
|        482|      125|
|        324|      124|
|        234|      123|
+-----------+---------+
only showing top 10 rows


In [ ]:

# Calculate skew ratio
total_rows    = df.count()
top_key_count = skew_distribution.first()["row_count"]
avg_count     = total_rows / df.select("customer_id").distinct().count()

print(f"\nTotal rows         : {total_rows:,}")
print(f"Top key row count  : {top_key_count:,}")
print(f"Average per key    : {avg_count:,.0f}")
print(f"Skew ratio         : {top_key_count / avg_count:.1f}x — {'SEVERE' if top_key_count/avg_count > 10 else 'MODERATE'}")


Total rows         : 2,000,000
Top key row count  : 1,600,514
Average per key    : 2,000
Skew ratio         : 800.3x — SEVERE


In [ ]:
import time

# Create a small "customer details" table to join with
customer_data = [(i, f"Customer_{i}", f"Region_{i % 10}") for i in range(1, 1001)]
customer_df = spark.createDataFrame(customer_data, ["customer_id", "name", "region"])

# ---- SKEWED JOIN — no optimization ----
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)  # force shuffle join (disable broadcast)
spark.conf.set("spark.sql.shuffle.partitions", "8")          # small number for CE

start = time.time()

skewed_result = df.join(customer_df, on="customer_id", how="inner") \
                  .groupBy("customer_id", "region") \
                  .agg(
                      F.count("*").alias("total_orders"),
                      F.sum("amount").alias("total_revenue"),
                      F.avg("amount").alias("avg_order_value")
                  )

skewed_result.show(5)
elapsed = time.time() - start
print(f"\nSkewed join time: {elapsed:.2f} seconds")

+-----------+--------+------------+------------------+------------------+
|customer_id|  region|total_orders|     total_revenue|   avg_order_value|
+-----------+--------+------------+------------------+------------------+
|         42|Region_2|         112|265164.49999999994| 2367.540178571428|
|         91|Region_1|          83|199501.66999999995|2403.6345783132524|
|        267|Region_7|          93|         234846.75| 2525.233870967742|
|        271|Region_1|         105|258478.54000000004|2461.7003809523812|
|        325|Region_5|          89|         222770.28| 2503.036853932584|
+-----------+--------+------------+------------------+------------------+
only showing top 5 rows

Skewed join time: 6.00 seconds


In [ ]:
# See how many rows are in each partition AFTER a shuffle
from pyspark.sql.functions import spark_partition_id

partitioned = df.withColumn("partition_id", spark_partition_id())

partition_sizes = partitioned.groupBy("partition_id") \
    .agg(F.count("*").alias("rows_in_partition")) \
    .orderBy("partition_id")

partition_sizes.show()

# Standard deviation tells you how skewed partitions are
stats = partition_sizes.select(
    F.mean("rows_in_partition").alias("mean"),
    F.stddev("rows_in_partition").alias("stddev"),
    F.max("rows_in_partition").alias("max"),
    F.min("rows_in_partition").alias("min")
)
stats.show()

+------------+-----------------+
|partition_id|rows_in_partition|
+------------+-----------------+
|           0|          1000448|
|           1|           999552|
+------------+-----------------+

+---------+-----------------+-------+------+
|     mean|           stddev|    max|   min|
+---------+-----------------+-------+------+
|1000000.0|633.5676759431466|1000448|999552|
+---------+-----------------+-------+------+



In [ ]:
# Serverless-safe cluster/config inspection
print("Spark Version:", spark.version)

# Safe config reads — serverless may return "not set" for JVM-level configs, that's expected
configs_to_check = [
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.files.maxPartitionBytes",
    "spark.sql.autoBroadcastJoinThreshold"
]

print("\n--- Spark SQL Configurations ---")
for cfg in configs_to_check:
    try:
        val = spark.conf.get(cfg)
        print(f"{cfg}: {val}")
    except Exception:
        print(f"{cfg}: not set")

Spark Version: 4.0.2

--- Spark SQL Configurations ---
spark.sql.shuffle.partitions: 8
spark.sql.adaptive.enabled: false
spark.sql.adaptive.skewJoin.enabled: true
spark.sql.files.maxPartitionBytes: 134217728b
spark.sql.autoBroadcastJoinThreshold: -1


In [ ]:
# If the RIGHT table is small (< a few hundred MB), broadcast it
# This eliminates the shuffle entirely — no skew possible

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)  # 10MB auto threshold

start = time.time()

broadcast_result = df.join(F.broadcast(customer_df), on="customer_id", how="inner") \
                     .groupBy("customer_id", "region") \
                     .agg(
                         F.count("*").alias("total_orders"),
                         F.sum("amount").alias("total_revenue"),
                         F.avg("amount").alias("avg_order_value")
                     )

broadcast_result.show(5)
elapsed = time.time() - start
print(f"Broadcast join time: {elapsed:.2f} seconds")

+-----------+--------+------------+------------------+------------------+
|customer_id|  region|total_orders|     total_revenue|   avg_order_value|
+-----------+--------+------------+------------------+------------------+
|        900|Region_0|         100|249132.40999999997|         2491.3241|
|        877|Region_7|         107|         276478.56|2583.9117757009344|
|        296|Region_6|          89|199902.50999999995|2246.0956179775276|
|        805|Region_5|          94|251145.29999999993| 2671.758510638297|
|        409|Region_9|          97|239033.46999999997|2464.2625773195873|
+-----------+--------+------------+------------------+------------------+
only showing top 5 rows
Broadcast join time: 2.23 seconds


In [ ]:
# Serverless-safe cluster/config inspection
print("Spark Version:", spark.version)

# Safe config reads — serverless may return "not set" for JVM-level configs, that's expected
configs_to_check = [
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.files.maxPartitionBytes",
    "spark.sql.autoBroadcastJoinThreshold"
]

print("\n--- Spark SQL Configurations ---")
for cfg in configs_to_check:
    try:
        val = spark.conf.get(cfg)
        print(f"{cfg}: {val}")
    except Exception:
        print(f"{cfg}: not set")

Spark Version: 4.0.2

--- Spark SQL Configurations ---
spark.sql.shuffle.partitions: 8
spark.sql.adaptive.enabled: false
spark.sql.adaptive.skewJoin.enabled: true
spark.sql.files.maxPartitionBytes: 134217728b
spark.sql.autoBroadcastJoinThreshold: 10485760


In [ ]:
import math
from pyspark.sql.functions import col
# SALTING: Break the hot key into N sub-keys to spread load
SALT_FACTOR = 10  # split hot partition into 10 sub-partitions

# Step A: Add a random salt to the large table
df_salted = df.withColumn(
    "salt",
    (F.rand() * SALT_FACTOR).cast(IntegerType())
).withColumn(
    "salted_customer_id",
    F.concat(F.col("customer_id").cast(StringType()), F.lit("_"), F.col("salt"))
)

# Step B: Explode the small table to match all salt values
from pyspark.sql.functions import array, explode, lit

salt_array = array(*[lit(i) for i in range(SALT_FACTOR)])

customer_salted = customer_df.withColumn("salt_exploded", explode(salt_array)) \
    .withColumn(
        "salted_customer_id",
        F.concat(F.col("customer_id").cast(StringType()), F.lit("_"), F.col("salt_exploded").cast(StringType()))
    )

# Step C: Join on salted key
start = time.time()

# Alias DataFrames to resolve ambiguous column names
salted_result = df_salted.alias("df_s").join(customer_salted.alias("cust_s"), on="salted_customer_id", how="inner") \
    .groupBy(F.col("df_s.customer_id"), F.col("cust_s.region")) \
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("avg_order_value")
    )

salted_result.show(5)
elapsed = time.time() - start
print(f"Salted join time: {elapsed:.2f} seconds")

+-----------+--------+------------+------------------+------------------+
|customer_id|  region|total_orders|     total_revenue|   avg_order_value|
+-----------+--------+------------+------------------+------------------+
|        118|Region_8|          88|         201744.39|2292.5498863636367|
|        139|Region_9|          97|221451.74999999997|2283.0077319587626|
|        144|Region_4|          90|238573.91000000003| 2650.821222222223|
|        148|Region_8|         106|276244.39999999997|2606.0792452830187|
|        163|Region_3|         113|         281189.97|2488.4068141592916|
+-----------+--------+------------+------------------+------------------+
only showing top 5 rows
Salted join time: 8.66 seconds


In [ ]:
# Databricks has AQE built in — let it auto-coalesce skewed partitions
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")       # 5x median = skewed
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256mb")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.minPartitionNum", "1")

start = time.time()

aqe_result = df.join(customer_df, on="customer_id", how="inner") \
               .groupBy("customer_id", "region") \
               .agg(
                   F.count("*").alias("total_orders"),
                   F.sum("amount").alias("total_revenue"),
                   F.avg("amount").alias("avg_order_value")
               )

aqe_result.show(5)
elapsed = time.time() - start
print(f"AQE join time: {elapsed:.2f} seconds")

+-----------+--------+------------+------------------+------------------+
|customer_id|  region|total_orders|     total_revenue|   avg_order_value|
+-----------+--------+------------+------------------+------------------+
|        634|Region_4|         112|294830.69999999995|2632.4169642857137|
|        788|Region_8|         103| 269999.0399999999|2621.3499029126206|
|        271|Region_1|         105|258478.54000000004|2461.7003809523812|
|        501|Region_1|          82|208523.56999999998| 2542.970365853658|
|        459|Region_9|          90|237533.64000000013|2639.2626666666683|
+-----------+--------+------------+------------------+------------------+
only showing top 5 rows
AQE join time: 2.44 seconds


In [ ]:
# Serverless-safe cluster/config inspection
print("Spark Version:", spark.version)

# Safe config reads — serverless may return "not set" for JVM-level configs, that's expected
configs_to_check = [
    "spark.sql.adaptive.coalescePartitions.minPartitionNum",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.enabled"
]

print("\n--- Spark SQL Configurations ---")
for cfg in configs_to_check:
    try:
        val = spark.conf.get(cfg)
        print(f"{cfg}: {val}")
    except Exception:
        print(f"{cfg}: not set")

Spark Version: 4.0.2

--- Spark SQL Configurations ---
spark.sql.adaptive.coalescePartitions.minPartitionNum: 1
spark.sql.adaptive.coalescePartitions.enabled: true
spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes: 256mb
spark.sql.adaptive.skewJoin.enabled: true
spark.sql.adaptive.enabled: true


In [ ]:
# Summary comparison
print("=" * 50)
print("PERFORMANCE SUMMARY")
print("=" * 50)
print(f"{'Method':<25} | Notes")
print("-" * 50)
print(f"{'Skewed (no fix)':<25} | Straggler tasks, disk spill risk")
print(f"{'Broadcast join':<25} | Best if small table < 100MB")
print(f"{'Salting':<25} | Best for large-large joins")
print(f"{'AQE (Databricks)':<25} | Automatic, recommended default")

PERFORMANCE SUMMARY
Method                    | Notes
--------------------------------------------------
Skewed (no fix)           | Straggler tasks, disk spill risk
Broadcast join            | Best if small table < 100MB
Salting                   | Best for large-large joins
AQE (Databricks)          | Automatic, recommended default


In [ ]:
executor_memory = spark.conf.get("spark.executor.memory", "Not Set")
print(f"Executor Memory: {executor_memory}")

Executor Memory: Not Set


In [ ]:
# Run this in a code cell
!free -h


               total        used        free      shared  buff/cache   available
Mem:            12Gi       3.6Gi       5.3Gi       2.0Mi       3.8Gi       8.8Gi
Swap:             0B          0B          0B


In [ ]:
from pyspark.sql import functions as F

# Create a highly skewed dataset (90% of rows have ID 1)
data = [(1,) for _ in range(900000)] + [(i,) for i in range(2, 100001)]
df_skewed = spark.createDataFrame(data, ["id"])

# Create a small lookup table to join against
df_lookup = spark.createDataFrame([(i, f"Name_{i}") for i in range(1, 100001)], ["id", "name"])

# Disable automatic optimizations to see the skew clearly
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# Perform a join to trigger shuffle
df_joined = df_skewed.join(df_lookup, "id")
df_joined.write.format("noop").mode("overwrite").save()


In [ ]:
df_joined.withColumn("partition_id", F.spark_partition_id()) \
         .groupBy("partition_id").count() \
         .orderBy(F.desc("count")).show()


+------------+------+
|partition_id| count|
+------------+------+
|           5|912484|
|           7| 12678|
|           3| 12568|
|           2| 12515|
|           0| 12475|
|           1| 12455|
|           4| 12420|
|           6| 12404|
+------------+------+

